In [ ]:
import json
#import torch
from transformers import BertTokenizer, TFBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/Investopedia/train_items.jl'  # Replace with your actual path

In [ ]:
# Load data from JSON Lines file
#def load_data(file_path):
#    texts = []
#    labels = []
#    with open(file_path, 'r') as f:
#        for line in f:
#            st = r'%s' % line
#            data = json.loads(st)
#            texts.append(data['text'])
#            labels.append(data['label'])
#    return texts, labels

In [ ]:
def load_data_with_cleaning(file_path):
        texts = []
        labels = []
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    # Example cleaning: Replace literal backslashes with escaped ones
                    # This is a simplification and might not work for all issues!
                    cleaned_line = line.replace('\\"', '/')
                    #cleaned_line = cleaned_line1.replace('','')
                    # Ensure the cleaned line is a valid JSON string
                    data = json.loads(cleaned_line)
                    #combined_text = f"{data['title']} {data['text']}"
                    texts.append(data['title'])
                    #labels.append(data['label'])
                    # Convert label to integer
                    labels.append(int(data['label']))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line: {e}")
                    print(f"Problematic line content (first 200 chars): {line[:1535]}")
                    # Decide how to handle errors (skip line, raise error, etc.)
                    continue # Skip the problematic line

        return texts, labels

In [ ]:
texts, labels = load_data_with_cleaning(data_dir)

In [ ]:
print(texts)

['Eli Lilly (LLY) Option Traders Confident After Earnings', 'Eli Lilly Stock Drops as Lowered Profit Outlook Outweighs Solid Q1 Results', 'Eli Lilly (LLY) Option Traders Prepped for Earnings Beat', "UnitedHealth Loses $120 Billion in Value on Stock's Worst Day Since 1998", 'Health Insurance Stocks Finish Lower After Rough UnitedHealth Forecast ', 'Eli Lilly Stock Soars on Oral Weight-Loss Drug Trial Results', "Pfizer Halts Development of Obesity Drug After Patient Sustains 'Liver Injury'", "Pharma Stocks Sink as Trump Says 'Major' Tariffs Coming to Industry", 'Hims & Hers Stock Surges as Telehealth Platform Adds Eli Lilly Weight-Loss Drugs', 'Johnson & Johnson (JNJ) Surges to All-Time High', 'S&P 500 Gains & Losses Today: Johnson & Johnson Drops as Judge Rejects Liability Settlement', 'Watch These Johnson & Johnson Levels as Stock Plunges After Judge Rejects Talc Settlement', 'J&J Stock Slumps as Company Fails to Settle Talc Cases', 'Talc Lawsuits Keeping Lid on Johnson & Johnson (JNJ)

In [ ]:
#load_data(data_dir)

In [ ]:
#Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [ ]:
print(f"Number of training samples: {len(train_texts)}")

Number of training samples: 30


In [ ]:
# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Tokenize data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=100)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=100)

In [ ]:
# Convert to TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(4)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(4)

In [ ]:
# Optimizer and loss
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

In [ ]:
# Compile model
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [ ]:
# Train model
model.fit(train_dataset, epochs=5, validation_data=val_dataset
    )

Epoch 1/5
8/8 [==============================] - 60s 653ms/step - loss: 0.7137 - accuracy: 0.5333 - val_loss: 0.7125 - val_accuracy: 0.3750
Epoch 2/5
8/8 [==============================] - 1s 80ms/step - loss: 0.4464 - accuracy: 0.8667 - val_loss: 0.6352 - val_accuracy: 0.5000
Epoch 3/5
8/8 [==============================] - 1s 123ms/step - loss: 0.2910 - accuracy: 1.0000 - val_loss: 0.8223 - val_accuracy: 0.5000
Epoch 4/5
8/8 [==============================] - 1s 128ms/step - loss: 0.1365 - accuracy: 1.0000 - val_loss: 0.8903 - val_accuracy: 0.5000
Epoch 5/5
8/8 [==============================] - 1s 120ms/step - loss: 0.0515 - accuracy: 1.0000 - val_loss: 0.2860 - val_accuracy: 0.8750


In [ ]:
# Evaluate model
loss, accuracy = model.evaluate(val_dataset)
print(f"Loss: {loss}, Accuracy: {accuracy}")

2/2 [==============================] - 0s 31ms/step - loss: 0.2860 - accuracy: 0.8750
Loss: 0.2859848141670227, Accuracy: 0.875


In [ ]:
# Make predictions
text = "This movie was great!"
predict_input = tokenizer(text, truncation=True, padding=True, return_tensors='tf')
output = model(predict_input)[0]
prediction_value = tf.argmax(output, axis=1).numpy()[0]
print(f"Predicted sentiment: {prediction_value}")

Predicted sentiment: 1


In [ ]:
#File to Apply model to
new_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/Investopedia/items.jl'  # Replace with your actual path

In [70]:
# Function to load data from a JSON Lines file (similar to your existing function)
def load_new_data_with_cleaning(file_path):
    content = []
    texts = []
    ids = [] # Assuming each item has a unique ID you want to keep
    with open(file_path, 'r') as f:
        for line in f:
            try:
                cleaned_line = line.replace('\\"', '/')
                data = json.loads(cleaned_line)
                # Assuming you want to predict on the 'title' again
                texts.append(data['title'])
                content.append(data['text'])
                # Assuming an 'id' field exists to track the original item
                if 'url' in data:
                  ids.append(data['url'])
                else:
                  ids.append(None) # Or handle cases without an ID
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line: {e}")
                print(f"Problematic line content (first 200 chars): {line[:1535]}")
                continue
    return content, texts, ids

# Load the new data
new_content, new_texts, new_ids = load_new_data_with_cleaning(new_data_dir)

# Tokenize the new data
new_encodings = tokenizer(list(new_texts), truncation=True, padding=True, max_length=100, return_tensors='tf')

# Create a TensorFlow dataset for the new data
new_dataset = tf.data.Dataset.from_tensor_slices(
    dict(new_encodings)
).batch(4)

# Make predictions
predictions = model.predict(new_dataset)

# Get the predicted class (index with the highest probability)
predicted_labels = tf.argmax(predictions.logits, axis=1).numpy()

# Write the results back to a new file or overwrite the original
output_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/Investopedia/new_items_with_predictions.jl' # Define output file path

with open(output_data_dir, 'w') as outfile:
    for i, text in enumerate(new_texts):
        result = {
            'url': new_ids[i], # Include the original ID if available
            'title': new_texts[i],
            'text': new_content[i],
            'predicted_label': int(predicted_labels[i]) # Ensure it's an integer
        }
        outfile.write(json.dumps(result) + '\n')

print(f"Predictions written to {output_data_dir}")

50/50 [==============================] - 2s 37ms/step
Predictions written to /content/drive/MyDrive/Healthcare_Sentiment/Investopedia/new_items_with_predictions.jl
